# 03 · Results and model decision

**Question:** What did the completed experiments establish?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

**Current decision:** retain the lexical rule/example reference. No semantic candidate has established an overall replacement, and no leaderboard or medal result is claimed.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Compare model quality
Higher rule macro AUC is better. Lower log loss and Brier score are better. Familiar-rule and held-out-rule results answer different questions; do not pool their conclusions.

In [2]:
records = baseline["results"] + semantic["results"]
display(metric_table(records))

,Model,Validation,Rule macro AUC,Log loss,Brier,Average precision
0,Comment-only TF-IDF,Familiar rules,0.7281,0.6148,0.2131,0.7170
1,Rule/example TF-IDF,Familiar rules,0.7287,0.6162,0.2139,0.7202
2,Comment-only TF-IDF,Held-out rule,0.6041,0.6731,0.2400,0.6108
3,Rule/example TF-IDF,Held-out rule,0.6156,0.6736,0.2405,0.6241
4,Frozen semantic margin,Familiar rules,0.6351,0.6786,0.2423,0.6217
5,Semantic classifier,Familiar rules,0.6013,0.6742,0.2409,0.6083
6,Frozen semantic margin,Held-out rule,0.6351,0.6786,0.2423,0.6217
7,Semantic classifier,Held-out rule,0.5858,0.8264,0.2976,0.5321


![Recorded semantic and lexical comparison](../reports/semantic/comparison.svg)

## Does the semantic margin improve transfer?
These paired bootstrap intervals resample normalized comment groups and condition on the two observed rules and fixed predictions. They do not quantify transfer to arbitrary new policies.

In [3]:
intervals = pd.DataFrame(semantic["uncertainty"])
display(intervals.loc[intervals.protocol == "heldout_rule", ["model", "observed_delta", "ci_lower", "ci_upper", "draws"]].round(4))
print("Evidence verified for", baseline["training_rows"], "competition training rows.")
print("New training performed by this notebook: none.")

,model,observed_delta,ci_lower,ci_upper,draws
2,semantic_margin,0.0195,-0.0132,0.0491,500
3,semantic_classifier,-0.0298,-0.0640,0.0014,500


Evidence verified for 2029 competition training rows.
New training performed by this notebook: none.


## Decision
The frozen semantic margin has a **+0.0195** held-out AUC difference, with a paired 95% interval of **−0.0132 to +0.0491**. That interval includes no improvement, and probability losses worsen slightly. The learned semantic classifier performs worse. A negative experiment is retained rather than hidden.

The margin makes identical predictions in both validation protocols because it fits no fold labels. Those columns are not independent replications. The 10-row preview test checks inference plumbing, not generalization.

Continue to [04 · Semantic benchmark](04_semantic_benchmark.ipynb) for per-rule diagnostics, runtime, and the next controlled experiment. Full OOF verification is `uv run jigsaw review` after restoring private artifacts.